# Neural Rendering and Coordinate-Based Scene Modeling

This notebook presents a compact neural rendering workflow built around two reusable Python modules:

- `part1_code.py`: positional encoding and coordinate-based image fitting.
- `part2_code.py`: NeRF-style ray generation, sampling, radiance-field prediction, and volumetric rendering.

The notebook is organized as a project walkthrough rather than a step-by-step implementation template.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), Path.cwd() / "projects/neural_rendering"]:
    if (candidate / "part1_code.py").is_file() and (candidate / "part2_code.py").is_file():
        sys.path.insert(0, str(candidate))
        break

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from part1_code import model_2d, normalize_coord, positional_encoding, train_2d_model
from part2_code import (
    get_batches,
    get_rays,
    nerf_model,
    one_forward_pass,
    stratified_sampling,
    volumetric_rendering,
)

%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 1. Coordinate-Based Image Fitting

The first experiment treats an image as a continuous function from normalized 2D coordinates to RGB values. Positional encoding lets a small MLP represent sharper details and higher-frequency color variation.

In [ ]:
def find_data_dir():
    candidates = [
        Path("data"),
        Path("projects/neural_rendering/data"),
        Path("../data"),
    ]
    for candidate in candidates:
        if (candidate / "aurora.jpg").is_file() and (candidate / "lego_data.npz").is_file():
            return candidate
    raise FileNotFoundError("Could not find neural rendering data directory.")

DATA_DIR = find_data_dir()
painting_np = imageio.imread(DATA_DIR / "aurora.jpg")
if painting_np.ndim == 2:
    painting_np = np.stack([painting_np] * 3, axis=-1)
elif painting_np.shape[-1] == 4:
    painting_np = painting_np[..., :3]

painting = torch.from_numpy(np.asarray(painting_np, dtype=np.float32) / 255.0).to(device)
height_painting, width_painting = painting.shape[:2]

plt.figure(figsize=(5, 5))
plt.imshow(painting.detach().cpu().numpy())
plt.axis("off")
plt.title("Target image")

In [ ]:
encoded_coordinates = normalize_coord(
    height_painting,
    width_painting,
    num_frequencies=6,
).to(device)

print("encoded coordinate shape:", tuple(encoded_coordinates.shape))
print("target image shape:", tuple(painting.shape))

The training helper below can be run with different frequency counts to compare how positional encoding changes reconstruction quality. Training is left behind a flag so opening the notebook does not immediately start a long GPU job.

In [ ]:
RUN_IMAGE_FIT = False

if RUN_IMAGE_FIT:
    pred = train_2d_model(
        test_img=painting,
        num_frequencies=6,
        device=device,
        show=True,
    )

## 2. NeRF-Style View Synthesis

The second experiment models a 3D scene as a radiance field. For each camera pose, rays are generated from camera intrinsics, sampled between near/far bounds, evaluated by an MLP, and composited with differentiable volumetric rendering.

In [ ]:
data = np.load(DATA_DIR / "lego_data.npz")

images_np = data["images"]
poses = torch.from_numpy(data["poses"]).float().to(device)
intrinsics = torch.from_numpy(data["intrinsics"]).float().to(device)

height, width = images_np.shape[1:3]
images = torch.from_numpy(images_np[:100, ..., :3]).float().to(device)
test_image = torch.from_numpy(images_np[101, ..., :3]).float().to(device)
test_pose = poses[101]

print("training images:", tuple(images.shape))
print("poses:", tuple(poses.shape))
print("intrinsics:\n", intrinsics.detach().cpu().numpy())

plt.figure(figsize=(5, 5))
plt.imshow(test_image.detach().cpu().numpy())
plt.axis("off")
plt.title("Held-out view")

### 2.1 Camera Rays and Stratified Samples

The ray utilities expose the geometric part of the pipeline: every pixel becomes a ray in world coordinates, and each ray receives evenly spaced samples in depth.

In [ ]:
ray_origins, ray_directions = get_rays(
    height,
    width,
    intrinsics,
    poses[0, :3, :3],
    poses[0, :3, 3],
)
ray_points, depth_points = stratified_sampling(
    ray_origins,
    ray_directions,
    near=0.667,
    far=2.0,
    samples=64,
)

print("ray origins:", tuple(ray_origins.shape))
print("ray directions:", tuple(ray_directions.shape))
print("sampled points:", tuple(ray_points.shape))
print("depth samples:", tuple(depth_points.shape))

In [ ]:
def plot_camera_poses(poses, every=10):
    selected = poses[::every]
    origins = selected[:, :3, 3].detach().cpu().numpy()
    directions = selected[:, :3, 2].detach().cpu().numpy()

    ax = plt.figure(figsize=(8, 6)).add_subplot(projection="3d")
    ax.quiver(
        origins[:, 0],
        origins[:, 1],
        origins[:, 2],
        directions[:, 0],
        directions[:, 1],
        directions[:, 2],
        length=0.12,
        normalize=True,
    )
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title("Camera pose sample")
    plt.show()

plot_camera_poses(poses)

### 2.2 Radiance Field and Volume Rendering

The rendering path batches encoded 3D points and viewing directions, predicts RGB/density values, and composites samples along each ray into an image.

In [ ]:
model = nerf_model(num_x_frequencies=10, num_d_frequencies=4).to(device)

ray_point_batches, ray_direction_batches = get_batches(
    ray_points,
    ray_directions,
    num_x_frequencies=10,
    num_d_frequencies=4,
)

print("point batches:", len(ray_point_batches), tuple(ray_point_batches[0].shape))
print("direction batches:", len(ray_direction_batches), tuple(ray_direction_batches[0].shape))

In [ ]:
RUN_UNTRAINED_PREVIEW = False

if RUN_UNTRAINED_PREVIEW:
    with torch.no_grad():
        preview = one_forward_pass(
            height=height,
            width=width,
            intrinsics=intrinsics,
            pose=test_pose,
            near=0.667,
            far=2.0,
            samples=64,
            model=model,
            num_x_frequencies=10,
            num_d_frequencies=4,
        )
    plt.figure(figsize=(5, 5))
    plt.imshow(preview.detach().cpu().numpy())
    plt.axis("off")
    plt.title("Untrained render preview")

### 2.3 Training Loop

This loop is ready for longer experiments after choosing a GPU runtime. It is disabled by default to keep the notebook lightweight for review.

In [ ]:
RUN_NERF_TRAINING = False

if RUN_NERF_TRAINING:
    num_x_frequencies = 10
    num_d_frequencies = 4
    learning_rate = 5e-4
    iterations = 3000
    samples = 64
    display = 25
    near = 0.667
    far = 2.0

    model = nerf_model(
        num_x_frequencies=num_x_frequencies,
        num_d_frequencies=num_d_frequencies,
    ).to(device)

    def weights_init(m):
        if isinstance(m, nn.Linear):
            torch.nn.init.xavier_uniform_(m.weight)

    model.apply(weights_init)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    psnrs = []
    iternums = []

    for i in range(iterations + 1):
        model.train()
        img_idx = torch.randint(0, images.shape[0], (1,), device=device).item()
        target = images[img_idx]
        pose = poses[img_idx]

        pred = one_forward_pass(
            height,
            width,
            intrinsics,
            pose,
            near,
            far,
            samples,
            model,
            num_x_frequencies,
            num_d_frequencies,
        )
        loss = F.mse_loss(pred, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if i % display == 0:
            model.eval()
            with torch.no_grad():
                test_rec_image = one_forward_pass(
                    height,
                    width,
                    intrinsics,
                    test_pose,
                    near,
                    far,
                    samples,
                    model,
                    num_x_frequencies,
                    num_d_frequencies,
                )
                test_loss = F.mse_loss(test_rec_image, test_image)
                psnr = -10.0 * torch.log10(test_loss)

            psnrs.append(psnr.item())
            iternums.append(i)
            print(f"Iteration {i} | train loss {loss.item():.4f} | held-out PSNR {psnr.item():.2f}")

    torch.save(model.state_dict(), "model_nerf.pt")